In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_11.pickle

In [ ]:
%%cudf.pandas.profile
### cell 11 ###

### cell 11 optimized
import pandas as pd

# Step 1: read all essays into Python lists
ids = []
texts = []
for path in tqdm(train_txt):
    with open(path, 'r') as f:
        text = f.read()
    ids.append(path.rsplit('/', 1)[-1].replace('.txt', ''))
    texts.append(text)

# Step 2: build a GPU-backed DataFrame via pandas shim
# (the %load_ext cudf.pandas extension makes pandas.DataFrame GPU-backed)
df_text = pd.DataFrame({'id': ids, 'text': texts})

# Step 3: vectorized string ops + cast to int64 to match original dtypes
df_text['essay_len'] = (
    df_text['text']
      .str.strip()
      .str.len()
      .astype('int64')
)
df_text['essay_words'] = (
    df_text['text']
      .str.split()
      .list.len()
      .astype('int64')
)

# Step 4: merge back into `train` and assign
tmp = train.merge(
    df_text[['id', 'essay_len', 'essay_words']],
    on='id', how='left'
)
train['essay_len'] = tmp['essay_len']
train['essay_words'] = tmp['essay_words']
